# ETS lab Tutorial 4: Orbital objects - I
### This notebook is licensed under [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/?ref=chooser-v1) © 2024 by André Moitinho (U. Lisbon)

## Setup


Here we import the libraries needed for this activity

While you should by now be familiar with the **numpy** (array manipulation) and **matplotlib/mpl_toolkits** (ploting), the other imports are less known:

The **ephem** library is used here for dealing with Two/Three Line Elements (TLEs) of orbital objects and computing their positions at different times.
The term "ephem" is short for the word ephemeris, which is the traditional term for a table giving the position of a planet, asteroid, or comet for a series of dates.

From the Astrophysics Source Code Library:  http://ascl.net/1112.014
_PyEphem provides scientific-grade astronomical computations for the Python programming language. Given a date and location on the Earth's surface, it can compute the positions of the Sun and Moon, of the planets and their moons, and of any asteroids, comets, or earth satellites whose orbital elements the user can provide. Additional functions are provided to compute the angular separation between two objects in the sky, to determine the constellation in which an object lies, and to find the times at which an object rises, transits, and sets on a particular day.
The numerical routines that lie behind PyEphem are those from the XEphem astronomy application (ascl:1112.013), whose author, Elwood Downey, generously gave permission for us to use them as the basis for PyEphem._

The **datetime** library is used for manipulating dates and times. We'll use it for setting times for orbit propagation.

Finally, the **csv** library handles CSV (Comma Separated Values) files and is used here for writing an output CSV file with results of orbit computations.

If you get errors check first if you haven't installed some of these libraries. 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import axes3d         #for 3D plots as the name suggests
import ephem
import datetime
import csv

### A quick overview of datetime 
More in https://docs.python.org/3/library/datetime.html

Set our date and inspect the result


In [2]:
mydate = datetime.datetime.today()
mydate

datetime.datetime(2026, 11, 20, 22, 3, 0, 954590)

You can set another date

In [3]:
# You can set the date
datetime.datetime(2020, 11, 8,15)

datetime.datetime(2020, 11, 8, 15, 0)

The timedelta object provides time intervals. This is handy for adding/subtracting time intervals to dates.
<br>i.e. for creating arrays of times

In [4]:
datetime.timedelta(minutes=20)

datetime.timedelta(seconds=1200)

In [5]:
print(mydate)
print(mydate + datetime.timedelta(hours=20))

2026-11-20 22:03:00.954590
2026-11-21 18:03:00.954590


### Overview of ephem

Set observer's coordinates and time
This example uses Lisbon

In [15]:
obs = ephem.Observer()
obs.lat = np.deg2rad(38.717)
obs.long = np.deg2rad(9.133)
obs.date = datetime.datetime.today()


# another example: Bahia
#obs.lat = np.deg2rad(-11.409874)
#obs.long = np.deg2rad(-41.280857)


Note that the date is stored in the "obs" object in another format
<br>Can you guess what this number means?

In [7]:
# note that the date is stored in the "obs" object in another format
# Can you guess what this number means?

obs.date


46345.418849705726

We can inspect the "obs" class with "help" 

In [8]:
help(obs)

Help on Observer in module ephem object:

class Observer(_libastro.Observer)
 |  A location on earth for which positions are to be computed.
 |
 |  An `Observer` instance allows you to compute the positions of
 |  celestial bodies as seen from a particular latitude and longitude on
 |  the Earth's surface.  The constructor takes no parameters; instead,
 |  set its attributes once you have created it.  Defaults:
 |
 |  `date` - the moment the `Observer` is created
 |  `lat` - zero latitude
 |  `lon` - zero longitude
 |  `elevation` - 0 meters above sea level
 |  `horizon` - 0 degrees
 |  `epoch` - J2000
 |  `temp` - 15 degrees Celsius
 |  `pressure` - 1010 mBar
 |
 |  Method resolution order:
 |      Observer
 |      _libastro.Observer
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __copy__ = copy(self)
 |
 |  __repr__(self)
 |      Return a useful textual representation of this Observer.
 |
 |  compute_pressure(self)
 |      Set the atmospheric pressure for the current ele

Load Satellite TLE data.

In [11]:
l1 = '0 STARLINK-1008'
l2 = '1 44714U 19074B   25307.57947445  .00020085  00000-0  13626-2 0  9998'
l3 = '2 44714  53.0552  32.7436 0001601  93.7776 266.3396 15.06423501329741'

# This one is old and will give an error when integrating the orbit. Try it.
#l1 = 'ISS (ZARYA)'         
#l2 = '1 25544U 98067A   20324.17323782  .00001608  00000-0  37253-4 0  9993'
#l3 = '2 25544  51.6464 301.5752 0001608  65.2533  28.5305 15.49051503256043'

orbitElement = ephem.readtle(l1,l2,l3)


Our TLE was stored in the "orbitElement" object.
<br>Let's inspect orbitElement and list its attributes.

In [12]:
attributes=dir(orbitElement)
print(attributes)

['M', '_M', '__class__', '__copy__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '_ap', '_decay', '_drag', '_e', '_epoch', '_inc', '_n', '_orbit', '_raan', 'a_dec', 'a_epoch', 'a_ra', 'alt', 'ap', 'az', 'catalog_number', 'circumpolar', 'compute', 'copy', 'dec', 'decay', 'drag', 'e', 'eclipsed', 'elevation', 'elong', 'epoch', 'g_dec', 'g_ra', 'ha', 'inc', 'mag', 'n', 'name', 'neverup', 'orbit', 'parallactic_angle', 'ra', 'raan', 'radius', 'range', 'range_velocity', 'rise_az', 'rise_time', 'set_az', 'set_time', 'size', 'sublat', 'sublong', 'transit_alt', 'transit_time', 'writedb']


Those are the names of the attributes. Some are obvious but others are not. 
<br>Since the object is documented, we can also evoke help.
<br>(or search for the pyephem package documentation - RTFM)

In [13]:
help(orbitElement)

Help on EarthSatellite object:

class EarthSatellite(Body)
 |  A satellite in orbit around the Earth, usually built by passing the text of a TLE entry to the `ephem.readtle()` routine. You can read and write its orbital parameters through the following attributes:
 |
 |  _ap -- argument of perigee at epoch (degrees)
 |  _decay -- orbit decay rate (revolutions per day-squared)
 |  _drag -- object drag coefficient (per earth radius)
 |  _e -- eccentricity
 |  _epoch -- reference epoch (mjd)
 |  _inc -- inclination (degrees)
 |  _M -- mean anomaly (degrees from perigee at epoch)
 |  _n -- mean motion (revolutions per day)
 |  _orbit -- integer orbit number of epoch
 |  _raan -- right ascension of ascending node (degrees)
 |
 |  Method resolution order:
 |      EarthSatellite
 |      Body
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(self, /, *args, **kwargs)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  -------------------------------

Compute the orbit

In [17]:
orbitElement.compute(obs)

ValueError: TLE elements are valid for a few weeks around their epoch, but you are asking about a date 382 days from the epoch

Check a result. e.g. altitude

In [13]:
orbitElement.alt

-0.18451732397079468

### Analysis - I

### Satellite tracks: Time series x-y line plot and trajectory in polar plot.

Make some datetimes covering a whole day (24h, starting at a time of your choice)
<br>This will provide the times for plotting 

In [9]:
midnight = datetime.datetime.replace(datetime.datetime.now(), hour=0)
#midnight = datetime.datetime.replace(datetime.datetime(2021, 11, 9, 0, 0), hour=0)

tstep = 20  # use with caution: 60/tstep must be an integer
dt  = [midnight + datetime.timedelta(minutes=tstep*x) for x in range(0, int(24*60/tstep))]

Compute satellite locations (altitude and azimuth - alt, az) at each datetime

In [10]:
sat_alt, sat_az = [], []
for date in dt:
    obs.date = date
    orbitElement.compute(obs)
    sat_alt.append(np.rad2deg(orbitElement.alt))
    sat_az.append(np.rad2deg(orbitElement.az))

NameError: name 'orbitElement' is not defined

Plot satellite tracks:
* alt vs time
* az vs time
* satellite track (alt and az) in a polar plot


In [16]:
# Plot
